In [1]:
# ITERATION 4
"""
stage_5.py -- Stage 5 (iteration 2): 5c2-raw hazard model, manifest-driven.

Frozen model contract (5c2-raw): grammar (bsince + ewm{2,6,18} per class, age, tod)
+ causal z-scored values @ {t-1, t-2, t-3}: signed & magnitude of every stream's
source column, leg amplitude, raw body signed & magnitude. Sign convention
confirming-positive (value * leg_dir).

Manifest-driven (iteration 2):
  - Fork axes (frame, session window, stream set) are READ from the Stage-0
    manifest, never redeclared here. Dropping a stream in Stage 0 (e.g. no-TICK)
    propagates automatically: grammar classes AND z-value channels both track the
    manifest stream set.
  - tod is derived from clock time (session_start), resolution-independent.
  - The manifest is baked into the model bundle so the booster carries its own
    contract; the worker asserts against it and prints it at startup.

Naming (iteration 2):
  SOURCE_PATH / src : the "source" oscillator file (HA OHLC + JMA + TICK + derivs),
                      Stage-0's input; rawer than bars/events. src is the primary
                      data frame, augmented in-place with raw body columns.
  RAW_FILE / raw1   : pure MNQ raw OHLC (transient; merged into src, then unused).
"""

import json
import numpy as np
import pandas as pd
import joblib
import lightgbm as lgb
from scipy.signal import lfilter
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import log_loss, roc_auc_score

from common import Featurizer, load_manifest, _expanding_z, _welford_check

In [2]:
# ---------------------------------------------------------------- CONFIG (per-run, explicit)
FRAME = 3
STAGE0_TAG = 'mnq-TICK-2025-2026-9-12am-i4'
#STAGE0_TAG = 'mnq-TICK-9-12am-i4'

MANIFEST_PATH = f"stage-0/manifest_{STAGE0_TAG}_{FRAME}s.json"
BARS_PATH = f"stage-0/bars_{STAGE0_TAG}_{FRAME}s.pqt"
EVENTS_PATH = f"stage-0/events_{STAGE0_TAG}_{FRAME}s.pqt"

ITER_DIR = "."                                   # iteration-2 root (encapsulated)
OUT_DIR = "stage-5"

VALID_FROM = "2025-07-01"
TRAIN_END = "2025-12-31"
TEST_FROM = "2026-01-01"

# for 2025-26 run
VALID_FROM = "2025-02-01"
TRAIN_END = "2025-02-28"
TEST_FROM = "2026-03-01"

# frozen 5c2 architecture constants (NOT fork axes -- stay in code)
TAUS = (2.0, 6.0, 18.0)
VALUE_LAGS = (1, 2)                               # t-2, t-3 (t-1 shift is implicit)
ZWARM = 20
TOD_BIN_MIN = 30
BODY_OPEN_COL = "rawOpen"
BODY_CLOSE_COL = "rawLast"
BODY_TAG = "raw"

LGBM_PARAMS = dict(
    objective="binary", metric="binary_logloss", learning_rate=0.05,
    num_leaves=127, min_data_in_leaf=1000, feature_fraction=0.9,
    bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0,
    num_threads=16, verbosity=-1,
)
NUM_ROUNDS = 8000
EARLY_STOP = 200
STAGE5_ANCHOR = 0.27402
# ----------------------------------------------------------------

In [3]:
# ---------------------------------------------------------------- grammar features
def build_grammar_features(fz, date_from=None, date_to=None):
    blocks = []
    for S in fz._selected(date_from, date_to):
        t = np.nonzero(~S["warm"])[0]
        n = S["n"]
        cols = []
        for c in fz.classes:
            P = S["P"][c]
            ind = np.diff(P).astype(np.float64)
            occ = np.where(ind > 0, np.arange(n), -1)
            last = np.maximum.accumulate(occ)
            lastm1 = np.concatenate(([-1], last[:-1]))
            bsince = np.where(lastm1 >= 0, np.arange(n) - lastm1, np.arange(n) + 1)
            cols.append(bsince[t])
            x = np.concatenate(([0.0], ind[:-1]))
            for tau in TAUS:
                a = np.exp(-1.0 / tau)
                s = lfilter([a], [1.0, -a], x)
                cols.append(s[t])
        lt = np.where(t > 0, S["lt_incl"][np.maximum(t - 1, 0)], -1)
        age = np.where(lt >= 0, t - lt, t + 1)
        cols.append(age)
        cols.append(S["tod"][t].astype(np.float64))
        blocks.append(np.stack(cols, 1).astype(np.float32))
    return np.concatenate(blocks), fz.grammar_names

In [4]:
# ---------------------------------------------------------------- value features
def value_base_names(manifest):
    """Value channels derived from the manifest stream set (source columns)."""
    cols = manifest["_stream_cols"]
    names = ([f"z_{c}_signed" for c in cols]
             + [f"z_{c}_mag" for c in cols]
             + ["z_leg_amp", f"z_body_{BODY_TAG}_signed", f"z_body_{BODY_TAG}_mag"])
    return cols, names


def build_value_features(fz, src, date_from=None, date_to=None):
    cols, base = value_base_names(fz.manifest)
    names = base + [f"{nm}_lag{L}" for L in VALUE_LAGS for nm in base]
    sv = src.set_index("timestamp")
    blocks = []
    for S in fz._selected(date_from, date_to):
        ts = pd.DatetimeIndex(S["timestamp"])
        r = sv.reindex(ts)
        leg_dir = S["leg_dir"]
        feats = []
        for c in cols:                                             # signed per stream col
            feats.append(_expanding_z(r[c].to_numpy(np.float64) * leg_dir, ZWARM))
        for c in cols:                                             # magnitude per stream col
            feats.append(_expanding_z(np.abs(r[c].to_numpy(np.float64)), ZWARM))
        feats.append(_expanding_z(np.abs(r["JMA"].to_numpy(np.float64) - S["leg_start_jma"]), ZWARM))
        bo = r[BODY_OPEN_COL].to_numpy(np.float64)
        bc = r[BODY_CLOSE_COL].to_numpy(np.float64)
        feats.append(_expanding_z((bc - bo) * leg_dir, ZWARM))     # confirming-positive
        feats.append(_expanding_z(np.abs(bc - bo), ZWARM))

        M = np.stack(feats, 1)
        M = np.concatenate([np.zeros((1, M.shape[1])), M[:-1]], 0)          # t-1 shift
        lagged = [M]
        for L in VALUE_LAGS:
            lagged.append(np.concatenate([np.zeros((L, M.shape[1])), M[:-L]], 0))
        M = np.concatenate(lagged, 1)
        t = np.nonzero(~S["warm"])[0]
        Mt = M[t]
        Mt[(t < ZWARM + max(VALUE_LAGS))] = 0.0
        blocks.append(Mt.astype(np.float32))
    return np.concatenate(blocks), names


def build_X(fz, src, date_from=None, date_to=None):
    Xg, gn = build_grammar_features(fz, date_from, date_to)
    Xv, vn = build_value_features(fz, src, date_from, date_to)
    assert len(Xg) == len(Xv), (len(Xg), len(Xv))
    return np.hstack([Xg, Xv]), gn + vn


def build_meta(fz, date_from=None, date_to=None):
    bi, ts, tg, dt = [], [], [], []
    for S in fz._selected(date_from, date_to):
        t = np.nonzero(~S["warm"])[0]
        bi.append(S["bar_index"][t])
        ts.append(S["timestamp"][t])
        tg.append(S["tgt"][t])
        dt.append(np.full(len(t), str(S["sess"])))
    return pd.DataFrame({"bar_index": np.concatenate(bi),
                         "timestamp": np.concatenate(ts),
                         "is_target": np.concatenate(tg),
                         "date": np.concatenate(dt)})

In [5]:
# ---------------------------------------------------------------- train / eval
def train(fz, src, train_end, valid_from):
    X, names = build_X(fz, src, None, train_end)
    meta = build_meta(fz, None, train_end)
    y = meta["is_target"].to_numpy().astype(np.int8)
    va = (meta["date"] >= valid_from).to_numpy()
    tr = ~va
    dtr = lgb.Dataset(X[tr], label=y[tr], feature_name=names)
    dva = lgb.Dataset(X[va], label=y[va], reference=dtr)
    booster = lgb.train(LGBM_PARAMS, dtr, num_boost_round=NUM_ROUNDS,
                        valid_sets=[dva], valid_names=["valid"],
                        callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False),
                                   lgb.log_evaluation(200)])
    p_va = booster.predict(X[va], num_iteration=booster.best_iteration)
    iso = IsotonicRegression(out_of_bounds="clip").fit(p_va, y[va])
    print(json.dumps(dict(n_train=int(tr.sum()), n_valid=int(va.sum()),
                          best_iteration=int(booster.best_iteration),
                          valid_logloss_cal=float(log_loss(y[va], iso.predict(p_va)))),
                     indent=2))
    imp = pd.DataFrame({"feature": names,
                        "gain": booster.feature_importance("gain")}
                       ).sort_values("gain", ascending=False)
    print(imp.head(20).to_string(index=False))
    return dict(booster=booster, iso=iso, feature_names=names,
                valid_from=valid_from, train_end=train_end,
                manifest=fz.manifest, tag=STAGE0_TAG, importance=imp)


def evaluate(fz, src, model, start, end=None, anchor=STAGE5_ANCHOR):
    X, _ = build_X(fz, src, start, end)
    meta = build_meta(fz, start, end)
    y = meta["is_target"].to_numpy().astype(np.int8)
    p = model["booster"].predict(X, num_iteration=model["booster"].best_iteration)
    p_cal = model["iso"].predict(p)
    ll_cal = log_loss(y, p_cal)
    ll_const = log_loss(y, np.full_like(p, y.mean(), dtype=np.float64))
    print(json.dumps(dict(n_rows=int(len(y)), holdout_logloss_cal=float(ll_cal),
                          holdout_logloss_const=float(ll_const),
                          skill=float(1 - ll_cal / ll_const),
                          auc=float(roc_auc_score(y, p)),
                          anchor=anchor, delta=float(ll_cal - anchor)), indent=2))
    out = meta[["bar_index", "timestamp", "is_target"]].copy()
    out["p"] = p.astype(np.float32)
    out["p_cal"] = p_cal.astype(np.float32)
    tbl = out.assign(bin=pd.qcut(out["p_cal"], 10, duplicates="drop")).groupby(
        "bin", observed=True).agg(mean_p=("p_cal", "mean"),
                                  realized=("is_target", "mean"), n=("p_cal", "size"))
    print(tbl.to_string())
    return out

In [6]:
# ---------------------------------------------------------------- run
manifest = load_manifest(MANIFEST_PATH,TOD_BIN_MIN)

SOURCE_PATH = manifest["source_file"]
RAW_FILE = manifest["raw_file"]

assert STAGE0_TAG == manifest["stage0_tag"]

bars = pd.read_parquet(BARS_PATH)
events = pd.read_parquet(EVENTS_PATH)

sess_lo = pd.Timestamp(manifest["session_start"]).time()
sess_hi = pd.Timestamp(manifest["session_end"]).time()

src = pd.read_parquet(SOURCE_PATH)
src = src[(src["timestamp"].dt.time >= sess_lo) & (src["timestamp"].dt.time < sess_hi)]

raw1 = pd.read_parquet(RAW_FILE)
raw1 = raw1[(raw1["timestamp"].dt.time >= sess_lo) & (raw1["timestamp"].dt.time < sess_hi)]
raw1 = raw1.rename(columns={"Open": "rawOpen", "High": "rawHigh", "Low": "rawLow", "Last": "rawLast"})

src = src.merge(raw1[["timestamp", "rawOpen", "rawHigh", "rawLow", "rawLast"]], on="timestamp", how="left")

###  assert day start exists in src and raw ###
src_min_time = src["timestamp"].dt.time.min()
print(f'SRC MIN TIME: {src_min_time}')
assert sess_lo == src_min_time

assert src[["rawOpen", "rawLast"]].notna().all().all(), "raw OHLC has gaps vs source timestamps"

SRC MIN TIME: 09:00:00


In [7]:
print('--------------------- !! VERIFY !! ---------------------')
print(f'FRAME: {FRAME}sec, STAGE0_TAG: {STAGE0_TAG}, BODY_TAG: {BODY_TAG}')
print(f'VALID_FROM: {VALID_FROM}, TRAIN_END: {TRAIN_END }, TEST_FROM: {TEST_FROM}')
print('----------------------- MANIFEST -----------------------')
print(json.dumps({k: v for k, v in manifest.items() if not k.startswith("_")}, indent=2))
print('--------------------------------------------------------')

#

fz = Featurizer(bars, events, manifest, TOD_BIN_MIN, TAUS)
#augment_featurizer(fz, bars)

S0 = fz.sessions[len(fz.sessions) // 2]
xchk = src.set_index("timestamp").reindex(pd.DatetimeIndex(S0["timestamp"]))["jmaD1"].to_numpy(np.float64)
print("welford max abs diff:", _welford_check(xchk, ZWARM))

model = train(fz, src, TRAIN_END, VALID_FROM)
pred = evaluate(fz, src, model, TEST_FROM)

print(f"-------------- {STAGE0_TAG} {FRAME}s --------------")
joblib_file = f"{OUT_DIR}/model_{BODY_TAG}_{STAGE0_TAG}_{FRAME}s.joblib"
importance_file = f"{OUT_DIR}/importance_{BODY_TAG}_{STAGE0_TAG}_{FRAME}s.csv"
pred_file = f"{OUT_DIR}/pred_{BODY_TAG}_{STAGE0_TAG}_{FRAME}s.pqt"

joblib.dump({k: v for k, v in model.items() if k != "importance"}, joblib_file)
model["importance"].to_csv(importance_file, index=False)
pred.to_parquet(pred_file, index=False)

print(f'    joblib_file: {joblib_file}')
print(f'importance_file: {importance_file}')
print(f'      pred_file: {pred_file}')

#

z = pred.timestamp >= TEST_FROM
print("holdout-window logloss:", log_loss(pred.is_target[z], pred.p_cal[z]))

--------------------- !! VERIFY !! ---------------------
FRAME: 3sec, STAGE0_TAG: mnq-TICK-2025-2026-9-12am-i4, BODY_TAG: raw
VALID_FROM: 2025-02-01, TRAIN_END: 2025-02-28, TEST_FROM: 2026-03-01
----------------------- MANIFEST -----------------------
{
  "frame_seconds": 3,
  "session_start": "09:00",
  "session_end": "12:00",
  "warmup_bars": 10,
  "streams": [
    {
      "name": "MNQ_D1",
      "column": "jmaD1"
    },
    {
      "name": "MNQ_D2",
      "column": "jmaD2"
    },
    {
      "name": "TICK_D1",
      "column": "tickJmaD1"
    },
    {
      "name": "TICK_D2",
      "column": "tickJmaD2"
    }
  ],
  "self_stream": "MNQ_JMA_SELF",
  "source_file": "data/mnq-tick-full-3sec.pqt",
  "raw_file": "data/mnq-ohlc-raw-3sec.pqt",
  "n_bars": 1428720,
  "n_sessions": 397,
  "date_min": "2025-01-02 00:00:00",
  "date_max": "2026-07-24 00:00:00",
  "created": "2026-07-25T16:13:56.159646",
  "from_year": 2025,
  "stage0_tag": "mnq-TICK-2025-2026-9-12am-i4"
}
----------------------

In [8]:
##########################################################
# RED / GREEN / STAGE5_ANCHOR - USED in TOD_SKILL and PROD 
##########################################################
BODY_TAG = 'raw'
pred_file = f"{OUT_DIR}/pred_{BODY_TAG}_{STAGE0_TAG}_{FRAME}s.pqt"

p = pd.read_parquet(pred_file)
p['date'] = p['timestamp'].dt.normalize()
#print(p.info())

print('==================================================================================')
print('================================== SUMMARY =======================================')
print('==================================================================================')
print(pred_file + '\n')

h = p[p["date"] >= "2026-01-01"]          # no-op if the parquet is holdout-only
GREEN = float(h["p_cal"].quantile(0.50))
RED   = float(h["p_cal"].quantile(0.90))
g = h[h["p_cal"] <  GREEN]
r = h[h["p_cal"] >= RED]
print(f"GREEN {GREEN:.8g}  {len(g)/len(h):.2%} of bars  wrong 1/{1/g['is_target'].mean():.0f}")
print(f"RED   {RED:.8g}    {len(r)/len(h):.2%} of bars  right {r['is_target'].mean():.1%}")

#

y = h["is_target"].values.astype(np.float64)
pc = np.clip(h["p_cal"].values.astype(np.float64), 1e-15, 1-1e-15)
a  = y.mean()
ll_model = -(y*np.log(pc) + (1-y)*np.log(1-pc)).mean()
ll_const = -(a*np.log(a) + (1-a)*np.log(1-a))
print(f"y.mean {a:.5f}  ll_model {ll_model:.5f}  ll_const {ll_const:.5f}  skill {1-ll_model/ll_const:.4f}  rows {len(h)}")

================================== SUMMARY =======================================
stage-5/pred_raw_mnq-TICK-2025-2026-9-12am-i4_3s.pqt

GREEN 0.0031821798  44.98% of bars  wrong 1/1214
RED   0.34836066    11.02% of bars  right 69.1%
y.mean 0.10336  ll_model 0.14181  ll_const 0.33240  skill 0.5734  rows 369697


In [9]:
# visual check
print(f"-------------- bars --------------")
print(bars.head())
print(f"\n-------------- events --------------")
print(events.head())
print(f"\n-------------- model --------------")
print(model)
print(f"\n-------------- pred --------------")
print(pred.head())

-------------- bars --------------
   bar_index           timestamp       date           jma        d1  \
0          0 2025-01-02 09:00:00 2025-01-02  21416.615234  0.218750   
1          1 2025-01-02 09:00:03 2025-01-02  21416.580078  0.085938   
2          2 2025-01-02 09:00:06 2025-01-02  21415.433594 -1.181641   
3          3 2025-01-02 09:00:09 2025-01-02  21413.441406 -3.138672   
4          4 2025-01-02 09:00:12 2025-01-02  21412.263672 -3.169922   

   jma_leg_dir  leg_id  leg_age   leg_amp  is_target  warm  
0            0       0        0  0.000000      False  True  
1           -1       0        1  0.035156      False  True  
2           -1       0        2  1.181641      False  True  
3           -1       0        3  3.173828      False  True  
4           -1       0        4  4.351562      False  True  

-------------- events --------------
        date        stream  event_bar  extremum_bar           timestamp  \
0 2025-01-02        MNQ_D1         10             9 2025-01

In [10]:
'''
2022 YEAR
--------------------- !! VERIFY !! ---------------------
FRAME: 3sec, STAGE0_TAG: mnq-TICK-9-12am-i4, BODY_TAG: raw
----------------------- MANIFEST -----------------------
{
  "frame_seconds": 3,
  "session_start": "09:00",
  "session_end": "12:00",
  "warmup_bars": 10,
  "streams": [
    {
      "name": "MNQ_D1",
      "column": "jmaD1"
    },
    {
      "name": "MNQ_D2",
      "column": "jmaD2"
    },
    {
      "name": "TICK_D1",
      "column": "tickJmaD1"
    },
    {
      "name": "TICK_D2",
      "column": "tickJmaD2"
    }
  ],
  "self_stream": "MNQ_JMA_SELF",
  "source_file": "data/mnq-tick-full-3sec.pqt",
  "raw_file": "data/mnq-ohlc-raw-3sec.pqt",
  "n_bars": 4159683,
  "n_sessions": 1156,
  "date_min": "2022-01-03 00:00:00",
  "date_max": "2026-07-24 00:00:00",
  "created": "2026-07-25T15:19:13.722136",
  "stage0_tag": "mnq-TICK-9-12am-i4"
}
--------------------------------------------------------
welford max abs diff: 7.105427357601002e-15
[200]	valid's binary_logloss: 0.133954
[400]	valid's binary_logloss: 0.130201
[600]	valid's binary_logloss: 0.129267
[800]	valid's binary_logloss: 0.128892
[1000]	valid's binary_logloss: 0.128686
[1200]	valid's binary_logloss: 0.128581
[1400]	valid's binary_logloss: 0.128533
[1600]	valid's binary_logloss: 0.128483
[1800]	valid's binary_logloss: 0.128463
{
  "n_train": 3175513,
  "n_valid": 452137,
  "best_iteration": 1782,
  "valid_logloss_cal": 0.12798059666338138
}
                feature         gain
      z_body_raw_signed 2.043427e+06
            z_jmaD1_mag 1.363387e+06
         z_jmaD2_signed 1.291190e+06
         z_jmaD1_signed 1.109470e+06
 z_body_raw_signed_lag1 1.063536e+06
         z_body_raw_mag 9.694371e+05
       z_jmaD1_mag_lag1 5.160419e+05
    z_jmaD1_signed_lag1 3.601341e+05
    z_body_raw_mag_lag1 3.338306e+05
      MNQ_D1|opp|bsince 2.955948e+05
     MNQ_D1|conf|bsince 2.790675e+05
 z_body_raw_signed_lag2 2.586942e+05
MNQ_JMA_SELF|all|bsince 2.316912e+05
            z_jmaD2_mag 2.246165e+05
       z_jmaD1_mag_lag2 1.797044e+05
        MNQ_D1|opp|ewm2 1.505032e+05
    z_jmaD1_signed_lag2 1.180588e+05
                    tod 1.136711e+05
  MNQ_JMA_SELF|all|ewm2 1.109254e+05
    z_jmaD2_signed_lag1 9.725392e+04
{
  "n_rows": 516883,
  "holdout_logloss_cal": 0.12440918988064581,
  "holdout_logloss_const": 0.3332510857587206,
  "skill": 0.6266803164424792,
  "auc": 0.9707044589962515,
  "anchor": 0.27402,
  "delta": -0.14961081011935418
}
                        mean_p  realized      n
bin                                            
(-0.001, 0.000124]    0.000089  0.000241  87086
(0.000124, 0.000248]  0.000194  0.000290  17232
(0.000248, 0.000591]  0.000455  0.000610  57419
(0.000591, 0.00111]   0.000918  0.000983  56973
(0.00111, 0.00233]    0.001998  0.001464  43712
(0.00233, 0.00635]    0.004152  0.003619  48080
(0.00635, 0.0206]     0.013795  0.012416  52917
(0.0206, 0.0945]      0.051340  0.049141  51139
(0.0945, 0.409]       0.215973  0.219244  51454
(0.409, 1.0]          0.739486  0.763146  50871
-------------- mnq-TICK-9-12am-i4 3s --------------
    joblib_file: stage-5/model_raw_mnq-TICK-9-12am-i4_3s.joblib
importance_file: stage-5/importance_raw_mnq-TICK-9-12am-i4_3s.csv
      pred_file: stage-5/pred_raw_mnq-TICK-9-12am-i4_3s.pqt
holdout-window logloss: 0.1242925226688385
'''

'\n2022 YEAR\n--------------------- !! VERIFY !! ---------------------\nFRAME: 3sec, STAGE0_TAG: mnq-TICK-9-12am-i4, BODY_TAG: raw\n----------------------- MANIFEST -----------------------\n{\n  "frame_seconds": 3,\n  "session_start": "09:00",\n  "session_end": "12:00",\n  "warmup_bars": 10,\n  "streams": [\n    {\n      "name": "MNQ_D1",\n      "column": "jmaD1"\n    },\n    {\n      "name": "MNQ_D2",\n      "column": "jmaD2"\n    },\n    {\n      "name": "TICK_D1",\n      "column": "tickJmaD1"\n    },\n    {\n      "name": "TICK_D2",\n      "column": "tickJmaD2"\n    }\n  ],\n  "self_stream": "MNQ_JMA_SELF",\n  "source_file": "data/mnq-tick-full-3sec.pqt",\n  "raw_file": "data/mnq-ohlc-raw-3sec.pqt",\n  "n_bars": 4159683,\n  "n_sessions": 1156,\n  "date_min": "2022-01-03 00:00:00",\n  "date_max": "2026-07-24 00:00:00",\n  "created": "2026-07-25T15:19:13.722136",\n  "stage0_tag": "mnq-TICK-9-12am-i4"\n}\n--------------------------------------------------------\nwelford max abs diff: 7

In [11]:
'''
2023 YEAR

--------------------- !! VERIFY !! ---------------------
FRAME: 3sec, STAGE0_TAG: mnq-TICK-2023-2026-9-12am-i4, BODY_TAG: raw
----------------------- MANIFEST -----------------------
{
  "frame_seconds": 3,
  "session_start": "09:00",
  "session_end": "12:00",
  "warmup_bars": 10,
  "streams": [
    {
      "name": "MNQ_D1",
      "column": "jmaD1"
    },
    {
      "name": "MNQ_D2",
      "column": "jmaD2"
    },
    {
      "name": "TICK_D1",
      "column": "tickJmaD1"
    },
    {
      "name": "TICK_D2",
      "column": "tickJmaD2"
    }
  ],
  "self_stream": "MNQ_JMA_SELF",
  "source_file": "data/mnq-tick-full-3sec.pqt",
  "raw_file": "data/mnq-ohlc-raw-3sec.pqt",
  "n_bars": 3242124,
  "n_sessions": 901,
  "date_min": "2023-01-03 00:00:00",
  "date_max": "2026-07-24 00:00:00",
  "created": "2026-07-25T15:38:33.837043",
  "from_year": 2023,
  "stage0_tag": "mnq-TICK-2023-2026-9-12am-i4"
}
--------------------------------------------------------
welford max abs diff: 6.217248937900877e-15
[200]	valid's binary_logloss: 0.134148
[400]	valid's binary_logloss: 0.130792
[600]	valid's binary_logloss: 0.130011
[800]	valid's binary_logloss: 0.129721
[1000]	valid's binary_logloss: 0.129553
[1200]	valid's binary_logloss: 0.12953
[1400]	valid's binary_logloss: 0.129535
{
  "n_train": 2260504,
  "n_valid": 452137,
  "best_iteration": 1305,
  "valid_logloss_cal": 0.12892757060035206
}
                feature         gain
      z_body_raw_signed 1.441392e+06
            z_jmaD1_mag 9.865296e+05
         z_jmaD2_signed 9.337501e+05
         z_jmaD1_signed 7.867419e+05
 z_body_raw_signed_lag1 7.526392e+05
         z_body_raw_mag 6.766327e+05
       z_jmaD1_mag_lag1 3.730699e+05
    z_jmaD1_signed_lag1 2.620065e+05
    z_body_raw_mag_lag1 2.251615e+05
     MNQ_D1|conf|bsince 2.167460e+05
      MNQ_D1|opp|bsince 2.164451e+05
 z_body_raw_signed_lag2 1.822361e+05
            z_jmaD2_mag 1.644062e+05
MNQ_JMA_SELF|all|bsince 1.536397e+05
       z_jmaD1_mag_lag2 1.135188e+05
        MNQ_D1|opp|ewm2 1.031032e+05
  MNQ_JMA_SELF|all|ewm2 8.909020e+04
    z_jmaD1_signed_lag2 8.705714e+04
                    tod 7.905055e+04
    z_jmaD2_signed_lag1 6.933419e+04
{
  "n_rows": 516883,
  "holdout_logloss_cal": 0.12515607865696074,
  "holdout_logloss_const": 0.3332510857587206,
  "skill": 0.6244390971089713,
  "auc": 0.9703308515843432,
  "anchor": 0.27402,
  "delta": -0.14886392134303925
}
                        mean_p  realized      n
bin                                            
(-0.001, 0.000139]    0.000083  0.000241  66257
(0.000139, 0.000358]  0.000331  0.000520  88532
(0.000358, 0.000729]  0.000729  0.000392  20409
(0.000729, 0.000885]  0.000857  0.000849  34170
(0.000885, 0.00289]   0.002006  0.001680  59518
(0.00289, 0.00799]    0.005588  0.004455  45569
(0.00799, 0.0244]     0.014122  0.012997  49090
(0.0244, 0.0943]      0.052338  0.050551  51354
(0.0943, 0.366]       0.218000  0.219672  50976
(0.366, 1.0]          0.735680  0.760547  51008
-------------- mnq-TICK-2023-2026-9-12am-i4 3s --------------
    joblib_file: stage-5/model_raw_mnq-TICK-2023-2026-9-12am-i4_3s.joblib
importance_file: stage-5/importance_raw_mnq-TICK-2023-2026-9-12am-i4_3s.csv
      pred_file: stage-5/pred_raw_mnq-TICK-2023-2026-9-12am-i4_3s.pqt
holdout-window logloss: 0.1251171976327896

==================================================================================
================================== SUMMARY =======================================
==================================================================================
stage-5/pred_raw_mnq-TICK-2023-2026-9-12am-i4_3s.pqt

GREEN 0.0028853551  48.40% of bars  wrong 1/1544
RED   0.36642411    10.30% of bars  right 74.6%
y.mean 0.10375  ll_model 0.12515  ll_const 0.33325  skill 0.6244  rows 516883
'''

'\n2023 YEAR\n\n--------------------- !! VERIFY !! ---------------------\nFRAME: 3sec, STAGE0_TAG: mnq-TICK-2023-2026-9-12am-i4, BODY_TAG: raw\n----------------------- MANIFEST -----------------------\n{\n  "frame_seconds": 3,\n  "session_start": "09:00",\n  "session_end": "12:00",\n  "warmup_bars": 10,\n  "streams": [\n    {\n      "name": "MNQ_D1",\n      "column": "jmaD1"\n    },\n    {\n      "name": "MNQ_D2",\n      "column": "jmaD2"\n    },\n    {\n      "name": "TICK_D1",\n      "column": "tickJmaD1"\n    },\n    {\n      "name": "TICK_D2",\n      "column": "tickJmaD2"\n    }\n  ],\n  "self_stream": "MNQ_JMA_SELF",\n  "source_file": "data/mnq-tick-full-3sec.pqt",\n  "raw_file": "data/mnq-ohlc-raw-3sec.pqt",\n  "n_bars": 3242124,\n  "n_sessions": 901,\n  "date_min": "2023-01-03 00:00:00",\n  "date_max": "2026-07-24 00:00:00",\n  "created": "2026-07-25T15:38:33.837043",\n  "from_year": 2023,\n  "stage0_tag": "mnq-TICK-2023-2026-9-12am-i4"\n}\n--------------------------------------

In [12]:
'''
2024 YEAR

--------------------- !! VERIFY !! ---------------------
FRAME: 3sec, STAGE0_TAG: mnq-TICK-2024-2026-9-12am-i4, BODY_TAG: raw
----------------------- MANIFEST -----------------------
{
  "frame_seconds": 3,
  "session_start": "09:00",
  "session_end": "12:00",
  "warmup_bars": 10,
  "streams": [
    {
      "name": "MNQ_D1",
      "column": "jmaD1"
    },
    {
      "name": "MNQ_D2",
      "column": "jmaD2"
    },
    {
      "name": "TICK_D1",
      "column": "tickJmaD1"
    },
    {
      "name": "TICK_D2",
      "column": "tickJmaD2"
    }
  ],
  "self_stream": "MNQ_JMA_SELF",
  "source_file": "data/mnq-tick-full-3sec.pqt",
  "raw_file": "data/mnq-ohlc-raw-3sec.pqt",
  "n_bars": 2342743,
  "n_sessions": 651,
  "date_min": "2024-01-02 00:00:00",
  "date_max": "2026-07-24 00:00:00",
  "created": "2026-07-25T15:56:37.051403",
  "from_year": 2024,
  "stage0_tag": "mnq-TICK-2024-2026-9-12am-i4"
}
--------------------------------------------------------
welford max abs diff: 5.329070518200751e-15
[200]	valid's binary_logloss: 0.134587
[400]	valid's binary_logloss: 0.131803
[600]	valid's binary_logloss: 0.13127
[800]	valid's binary_logloss: 0.131162
[1000]	valid's binary_logloss: 0.131148
{
  "n_train": 1363623,
  "n_valid": 452137,
  "best_iteration": 892,
  "valid_logloss_cal": 0.1304872616269679
}
                feature          gain
      z_body_raw_signed 880043.380616
            z_jmaD1_mag 617155.224207
         z_jmaD2_signed 567436.962610
 z_body_raw_signed_lag1 464423.769949
         z_jmaD1_signed 446966.299256
         z_body_raw_mag 410401.083701
       z_jmaD1_mag_lag1 211320.063996
    z_jmaD1_signed_lag1 145425.824411
    z_body_raw_mag_lag1 139670.620041
      MNQ_D1|opp|bsince 128191.137444
 z_body_raw_signed_lag2 115566.876045
            z_jmaD2_mag 111857.284429
     MNQ_D1|conf|bsince 109419.742700
  MNQ_JMA_SELF|all|ewm2  81706.048283
       z_jmaD1_mag_lag2  68961.885782
        MNQ_D1|opp|ewm2  63570.414256
MNQ_JMA_SELF|all|bsince  59744.642157
    z_jmaD1_signed_lag2  48280.638565
    z_jmaD2_signed_lag1  46564.017414
       MNQ_D1|conf|ewm2  46397.922999
{
  "n_rows": 516883,
  "holdout_logloss_cal": 0.12669289988701019,
  "holdout_logloss_const": 0.3332510857587206,
  "skill": 0.6198274955396905,
  "auc": 0.9696260547604681,
  "anchor": 0.27402,
  "delta": -0.1473271001129898
}
                        mean_p  realized      n
bin                                            
(-0.001, 4.72e-05]    0.000042  0.000269  51998
(4.72e-05, 0.000488]  0.000301  0.000315  63517
(0.000488, 0.000499]  0.000499  0.000721  49935
(0.000499, 0.000767]  0.000755  0.001157  45820
(0.000767, 0.00231]   0.001928  0.001334  47218
(0.00231, 0.00761]    0.005295  0.003786  54417
(0.00761, 0.0252]     0.013915  0.013608  49972
(0.0252, 0.101]       0.053272  0.049652  51055
(0.101, 0.393]        0.216201  0.216794  51399
(0.393, 1.0]          0.729658  0.754151  51552
-------------- mnq-TICK-2024-2026-9-12am-i4 3s --------------
    joblib_file: stage-5/model_raw_mnq-TICK-2024-2026-9-12am-i4_3s.joblib
importance_file: stage-5/importance_raw_mnq-TICK-2024-2026-9-12am-i4_3s.csv
      pred_file: stage-5/pred_raw_mnq-TICK-2024-2026-9-12am-i4_3s.pqt
holdout-window logloss: 0.12653735280036926

==================================================================================
================================== SUMMARY =======================================
==================================================================================
stage-5/pred_raw_mnq-TICK-2024-2026-9-12am-i4_3s.pqt

GREEN 0.0023088763  49.13% of bars  wrong 1/1468
RED   0.39337113    10.48% of bars  right 73.7%
y.mean 0.10375  ll_model 0.12668  ll_const 0.33325  skill 0.6199  rows 516883
'''

'\n2024 YEAR\n\n--------------------- !! VERIFY !! ---------------------\nFRAME: 3sec, STAGE0_TAG: mnq-TICK-2024-2026-9-12am-i4, BODY_TAG: raw\n----------------------- MANIFEST -----------------------\n{\n  "frame_seconds": 3,\n  "session_start": "09:00",\n  "session_end": "12:00",\n  "warmup_bars": 10,\n  "streams": [\n    {\n      "name": "MNQ_D1",\n      "column": "jmaD1"\n    },\n    {\n      "name": "MNQ_D2",\n      "column": "jmaD2"\n    },\n    {\n      "name": "TICK_D1",\n      "column": "tickJmaD1"\n    },\n    {\n      "name": "TICK_D2",\n      "column": "tickJmaD2"\n    }\n  ],\n  "self_stream": "MNQ_JMA_SELF",\n  "source_file": "data/mnq-tick-full-3sec.pqt",\n  "raw_file": "data/mnq-ohlc-raw-3sec.pqt",\n  "n_bars": 2342743,\n  "n_sessions": 651,\n  "date_min": "2024-01-02 00:00:00",\n  "date_max": "2026-07-24 00:00:00",\n  "created": "2026-07-25T15:56:37.051403",\n  "from_year": 2024,\n  "stage0_tag": "mnq-TICK-2024-2026-9-12am-i4"\n}\n--------------------------------------

In [13]:
'''
2025 - little val / test

--------------------- !! VERIFY !! ---------------------
FRAME: 3sec, STAGE0_TAG: mnq-TICK-2025-2026-9-12am-i4, BODY_TAG: raw
----------------------- MANIFEST -----------------------
{
  "frame_seconds": 3,
  "session_start": "09:00",
  "session_end": "12:00",
  "warmup_bars": 10,
  "streams": [
    {
      "name": "MNQ_D1",
      "column": "jmaD1"
    },
    {
      "name": "MNQ_D2",
      "column": "jmaD2"
    },
    {
      "name": "TICK_D1",
      "column": "tickJmaD1"
    },
    {
      "name": "TICK_D2",
      "column": "tickJmaD2"
    }
  ],
  "self_stream": "MNQ_JMA_SELF",
  "source_file": "data/mnq-tick-full-3sec.pqt",
  "raw_file": "data/mnq-ohlc-raw-3sec.pqt",
  "n_bars": 1428720,
  "n_sessions": 397,
  "date_min": "2025-01-02 00:00:00",
  "date_max": "2026-07-24 00:00:00",
  "created": "2026-07-25T16:13:56.159646",
  "from_year": 2025,
  "stage0_tag": "mnq-TICK-2025-2026-9-12am-i4"
}
--------------------------------------------------------
welford max abs diff: 6.217248937900877e-15
[200]	valid's binary_logloss: 0.13783
[400]	valid's binary_logloss: 0.136753
{
  "n_train": 297885,
  "n_valid": 78924,
  "best_iteration": 348,
  "valid_logloss_cal": 0.13554127100004792
}
                feature          gain
      z_body_raw_signed 206535.508734
            z_jmaD1_mag 129001.394998
         z_jmaD2_signed 124606.972374
 z_body_raw_signed_lag1 113866.231045
         z_jmaD1_signed  98305.763592
         z_body_raw_mag  85585.631457
       z_jmaD1_mag_lag1  37775.219674
    z_body_raw_mag_lag1  33232.274811
 z_body_raw_signed_lag2  31123.932161
    z_jmaD1_signed_lag1  29642.484629
            z_jmaD2_mag  25751.749852
      MNQ_D1|opp|bsince  19733.757475
        MNQ_D1|opp|ewm2  19428.235168
       MNQ_D1|conf|ewm2  19210.150557
  MNQ_JMA_SELF|all|ewm2  18043.505731
       z_jmaD1_mag_lag2  14232.963733
MNQ_JMA_SELF|all|bsince  12301.225323
    z_jmaD2_signed_lag1  10242.742052
       z_jmaD2_mag_lag1   9684.004770
    z_jmaD1_signed_lag2   9244.433621
{
  "n_rows": 139983,
  "holdout_logloss_cal": 0.12674337923676787,
  "holdout_logloss_const": 0.32896921549962693,
  "skill": 0.6147257151576495,
  "auc": 0.9691841720291485,
  "anchor": 0.27402,
  "delta": -0.1472766207632321
}
                        mean_p  realized      n
bin                                            
(-0.001, 0.0004]      0.000196  0.000156  19292
(0.0004, 0.000609]    0.000609  0.000532  20687
(0.000609, 0.000635]  0.000635  0.000787  25408
(0.000635, 0.00355]   0.003239  0.002023   5933
(0.00355, 0.00589]    0.005348  0.004964  17525
(0.00589, 0.028]      0.021049  0.017401  11149
(0.028, 0.0915]       0.060314  0.056225  13037
(0.0915, 0.377]       0.215788  0.221177  13288
(0.377, 1.0]          0.737019  0.750000  13664
-------------- mnq-TICK-2025-2026-9-12am-i4 3s --------------
    joblib_file: stage-5/model_raw_mnq-TICK-2025-2026-9-12am-i4_3s.joblib
importance_file: stage-5/importance_raw_mnq-TICK-2025-2026-9-12am-i4_3s.csv
      pred_file: stage-5/pred_raw_mnq-TICK-2025-2026-9-12am-i4_3s.pqt
holdout-window logloss: 0.12631258368492126

==================================================================================
================================== SUMMARY =======================================
==================================================================================
stage-5/pred_raw_mnq-TICK-2025-2026-9-12am-i4_3s.pqt

GREEN 0.0035492459  49.60% of bars  wrong 1/1736
RED   0.37745097    10.02% of bars  right 74.1%
y.mean 0.10178  ll_model 0.12671  ll_const 0.32897  skill 0.6148  rows 139983
'''

'\n2025 - little val / test\n\n--------------------- !! VERIFY !! ---------------------\nFRAME: 3sec, STAGE0_TAG: mnq-TICK-2025-2026-9-12am-i4, BODY_TAG: raw\n----------------------- MANIFEST -----------------------\n{\n  "frame_seconds": 3,\n  "session_start": "09:00",\n  "session_end": "12:00",\n  "warmup_bars": 10,\n  "streams": [\n    {\n      "name": "MNQ_D1",\n      "column": "jmaD1"\n    },\n    {\n      "name": "MNQ_D2",\n      "column": "jmaD2"\n    },\n    {\n      "name": "TICK_D1",\n      "column": "tickJmaD1"\n    },\n    {\n      "name": "TICK_D2",\n      "column": "tickJmaD2"\n    }\n  ],\n  "self_stream": "MNQ_JMA_SELF",\n  "source_file": "data/mnq-tick-full-3sec.pqt",\n  "raw_file": "data/mnq-ohlc-raw-3sec.pqt",\n  "n_bars": 1428720,\n  "n_sessions": 397,\n  "date_min": "2025-01-02 00:00:00",\n  "date_max": "2026-07-24 00:00:00",\n  "created": "2026-07-25T16:13:56.159646",\n  "from_year": 2025,\n  "stage0_tag": "mnq-TICK-2025-2026-9-12am-i4"\n}\n-----------------------

In [14]:
x = 1/0

ZeroDivisionError: division by zero

In [ ]:
#import json
import numpy as np
import pandas as pd
#import joblib
#import lightgbm as lgb
#from scipy.signal import lfilter
#from sklearn.isotonic import IsotonicRegression
#from sklearn.metrics import log_loss, roc_auc_score


FRAME = 3
STAGE0_TAG = 'mnq-TICK-9-12am'
BODY_TAG = "raw"

#MANIFEST_PATH = f"stage-0/manifest_{STAGE0_TAG}_{FRAME}s.json"
#BARS_PATH = f"stage-0/bars_{STAGE0_TAG}_{FRAME}s.pqt"
#EVENTS_PATH = f"stage-0/events_{STAGE0_TAG}_{FRAME}s.pqt"

OUT_DIR = "stage-5"


##########################################################
# RED / GREEN / STAGE5_ANCHOR - USED in TOD_SKILL and PROD 
##########################################################
pred_file = f"{OUT_DIR}/pred_{BODY_TAG}_{STAGE0_TAG}_{FRAME}s.pqt"

p = pd.read_parquet(pred_file)
p['date'] = p['timestamp'].dt.normalize()
#print(p.info())

print('==================================================================================')
print('================================== SUMMARY =======================================')
print('==================================================================================')
h = p[p["date"] >= "2026-01-01"]          # no-op if the parquet is holdout-only
GREEN = float(h["p_cal"].quantile(0.50))
RED   = float(h["p_cal"].quantile(0.90))
g = h[h["p_cal"] <  GREEN]
r = h[h["p_cal"] >= RED]
print(f"GREEN {GREEN:.8g}  {len(g)/len(h):.2%} of bars  wrong 1/{1/g['is_target'].mean():.0f}")
print(f"RED   {RED:.8g}    {len(r)/len(h):.2%} of bars  right {r['is_target'].mean():.1%}")

#

y = h["is_target"].values.astype(np.float64)
pc = np.clip(h["p_cal"].values.astype(np.float64), 1e-15, 1-1e-15)
a  = y.mean()
ll_model = -(y*np.log(pc) + (1-y)*np.log(1-pc)).mean()
ll_const = -(a*np.log(a) + (1-a)*np.log(1-a))
print(f"\ny.mean {a:.5f}  ll_model {ll_model:.5f}  ll_const {ll_const:.5f}  skill {1-ll_model/ll_const:.4f}  rows {len(h)}")